### Chains 

In Langchain, chains are sequences of modular components such as prompts,language models ,retrievers and parsers linked together to automate multi-step tasks by passing the output of one component as the input to the next.


In [ ]:
import os
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
from langchain.chat_models import init_chat_model

In [59]:
model = init_chat_model("groq:openai/gpt-oss-20b")

In [60]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [61]:
# creating a prompt template for the user to enter
prompt = PromptTemplate(
    template="Generate 5 interesting facts about {topic}",
    input_variables=["topic"]
)

#creating an object of strparser
parser = StrOutputParser()

In [62]:
#creating a simple chain with prompt,model and parser.

chain = prompt | model | parser

result = chain.invoke({'topic':'cricket'})

print(result)

Here are five fun and lesser‑known facts about cricket that even seasoned fans might find surprising:

1. **The Oldest International Test Match**  
   The first official Test match was played between England and Australia in 1877 at the Melbourne Cricket Ground. That game lasted just three days and was a 0‑0 draw because the pitch was heavily affected by rain – a stark contrast to the seven‑day format we’re used to today.

2. **The Longest Ever International Cricket Match**  
   The 1939–40 England vs. Australia “Test” at the Gabba in Brisbane stretched to 12 days (including a day of rain‑induced suspension). The match finished in a 0‑0 draw because each side could only bat once, but the sheer length made it a record that still stands.

3. **The “Biggest” Cricket Ball in the World**  
   The Guinness World Record for the largest cricket ball (by circumference) is 1.3 m (5 ft 4 in). It was made from a rubberized polymer and weighed 1.8 kg (4 lb). It’s a novelty item, not used in play, b

In [63]:
# chain.get_graph().print_ascii()  :- to visualise the chain

### 1. Sequential Chain

In [ ]:
"""
Here we are going to form a chain which does the following task

[topic] -> [llm] -> [report] -> [llm] -> [summary]

we give a topic to the llm and then ask it respond to it and then resend the output to the llm and ask it to summarize it 

"""

In [65]:
prompt1  = PromptTemplate(
    template ='Generate a detailed report on {topic}',
    input_variables=['topic']
)
prompt2 = PromptTemplate(
    template='Generate a 5 Line summary on the following text \n {text}',
    input_variables=['text']
)
chain = prompt1 | model  | parser | prompt2 | model | parser

In [66]:
result = chain.invoke({'topic':"unemployement in india"})

In [67]:
print(result)

- India’s overall unemployment fell modestly to **7.7 % (FY 2023‑24)** from 8.4 % the previous year, while **youth unemployment (15‑29) remains high at 12.3 %**.  
- Urban unemployment (7.9 %) slightly exceeds rural (7.2 %), and **86 % of the workforce is informal, with 65 % of jobs lacking job‑security or social protection**.  
- The decline is largely driven by a small dip in underemployment and a marginal shift to better‑matched roles, but **slow GDP growth (4.2 %) and a structural skill mismatch keep job creation sluggish**.  
- Key policy initiatives—**National Employment Policy 2024, Skill India 2025, and Digital Skill Initiative**—aim to boost employment, yet implementation gaps and uneven coverage persist.  
- Regional disparities persist, with the Northeast at 10.1 % unemployment and the North at 7.1 %, underscoring the need for targeted, inclusive strategies that address informality, gender gaps, and the youth bulge.


### 2. Parallel Chain

In [ ]:
"""
Let's understand parallel chaining with the following example  
user uploads a document we need to output two things to the user: 1) notes 2)quiz
here we run two models one for notes and the other for quiz and send these two outputs to a third model to merge the output 

"""


In [69]:
#for this example we are using the same model for both use cases.

model1 = model
model2 = model

prompt1 = PromptTemplate(
    template='Generate short and simple notes for the following text \n {text}',
    input_variables=['text']
)
prompt2 = PromptTemplate(
    template='Generate 5 short question answers from the following text \n {text}',
    input_variables=['text']
)
prompt3 = PromptTemplate(
    template='Merge thhe provided notes and quiz into a single document \n notes -> {notes} and quiz {quiz}',
    input_variables=['notes','quiz']
)

parser = StrOutputParser()

In [70]:
# for execution of parallel chains we need to import RunnableParallel
from langchain_core.runnables import RunnableParallel

In [71]:
parallel_chain = RunnableParallel(
    {
        'notes': prompt1 | model1 | parser,
        'quiz' : prompt2 | model2 | parser
    }
)
merge_chain = prompt3 | model1 | parser

chain = parallel_chain |  merge_chain

In [72]:
text = """
In LangChain, chains are sequences of modular components—such as language models, prompts, retrievers, and parsers—linked together to automate multi-step tasks by passing the output of one component as input to the next.  This structure allows developers to build complex, end-to-end workflows that abstract the boilerplate code required for coordinating API calls and data processing. 
The core types of chains include:
LLMChain: The foundational unit that combines a PromptTemplate, a Language Model, and an optional Output Parser to process a single input. 
Sequential Chains: These execute multiple sub-chains in a defined order, where the output of one step automatically becomes the input for the next, ideal for breaking down complex problems. 
Router Chains: These use a language model to intelligently select a specific Destination Chain from a set of options based on the input query, falling back to a Default Chain if needed. 
Map/Reduce and Parallel Chains: These handle larger datasets by mapping functions over data or dispatching inputs to multiple branches in parallel before merging results. 
Modern LangChain development increasingly utilizes LangChain Expression Language (LCEL) with the pipe operator (|) to compose these components into streamlined pipelines, replacing older explicit chain classes for many use cases. 
"""
 

result = chain.invoke({'text':text})

print(result)

# LangChain – Quick Reference & Practice Quiz  

---

## 1. Key Take‑aways  

| Concept | What it is | Why it matters |
|---------|------------|----------------|
| **Chain** | A sequence of modular parts (LLM, prompt, retriever, parser, etc.) that hand‑off output to the next part as input. | Lets you build full workflows without writing boilerplate API‑call glue code. |
| **LLMChain** | One prompt + LLM (+ optional parser) → processes a single input. | The simplest building block for a single‑step LLM call. |
| **Sequential Chains** | Runs sub‑chains in order; each step’s output feeds the next. | Enables multi‑step logic (e.g., “summarise → re‑write → translate”). |
| **Router Chain** | Uses an LLM to pick the right downstream chain; falls back to a default if no match. | Dynamically routes queries to the most appropriate workflow. |
| **Map/Reduce & Parallel Chains** | Split data across many branches (parallel) or apply a function to each item (map) and then combine (reduce). | Handle

### 3. Conditional Chain

In [ ]:
"""
Suppose you are a product manager and the customer gives a review about your product 
there are two conditions here  1) Positive review  2) Negative review
if the review is positive you send it to an LLM and ask it to draft a response accordingly and the same applies to the negative review  
here the both models aren't executed simultaneously but instead only one is executed.
"""


In [73]:
prompt1 = PromptTemplate(
    template='Classify the sentiment of the following feedback text into positive or negative.Return only one word Positive or Negative.\n {feedback}',
    input_variables=['feedback']
)

prompt2 = PromptTemplate(
    template='Write an appropriate response to this positive feedback \n {feedback}',
    input_variables =['feedback']
)

prompt3 = PromptTemplate(
    template='Write an appropriate response to this negative feedback \n {feedback}',
    input_variables =['feedback']
)

classifier_chain = prompt1 | model | parser


In [74]:
# just like you need RunnableParallel for parallel chaining you need RunnableBranch for conditional chaining
from langchain_core.runnables import RunnableBranch ,RunnableLambda

In [75]:
"""
branch_chain = RunnableBranch(
     (condition1,chain1),
     (condition2,chain2),
     defualt chain
)

"""
branch_chain = RunnableBranch(
     (lambda x: x=='Positive',prompt2 | model | parser),
     (lambda x: x=='Negative',prompt3 | model | parser),
     RunnableLambda(lambda x :"Could not find sentiment"
))


In [76]:
chain = classifier_chain | branch_chain

print(chain.invoke({'feedback':'the website is not working and the delivery was too late and the delivery guy was too rude.'}))

Absolutely—here’s a concise, empathetic reply you can adapt to any negative feedback you receive:

---

**Subject:** Thank you for sharing your experience

Dear [Customer’s Name],

Thank you for taking the time to let us know how we fell short of your expectations. I’m truly sorry that [briefly summarize the issue, e.g., “the delivery was delayed and the product didn’t meet your expectations”]. This is not the standard of service we strive to provide, and I understand how frustrating it must have been for you.

We’re already looking into what went wrong and will take the following steps to resolve the situation:

1. **Immediate Action:** [e.g., “We’ve issued a full refund and will ship a replacement at no extra cost.”]
2. **Root‑Cause Analysis:** Our team is reviewing the process that caused the delay so we can prevent it in the future.
3. **Follow‑Up:** I’ll personally keep you updated on the status and will reach out once the issue is fully resolved.

Your satisfaction is our top pri